## Learning targets

1. Make LR-ASPP emit two class-score channels at each pixel.
2. Freeze the visual backbone and optimize only the new classifier head.
3. Save a checkpoint plus validation predictions that Session 5 can inspect without retraining.

In [ ]:
import sys

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision.datasets import OxfordIIITPet
from torchvision.models.segmentation import LRASPP_MobileNet_V3_Large_Weights, lraspp_mobilenet_v3_large
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

SEED = 17
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 8
EPOCHS = 1
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_PATH = ARTIFACT_DIR / "session4_baseline.pt"
PREPARED_BASELINE = ARTIFACT_DIR / "instructor_baseline.pt"
print(f"torch={torch.__version__} | torchvision={torchvision.__version__} | device={device} | seed={SEED}")

torch=2.11.0+cpu | torchvision=0.26.0+cpu | device=cpu | seed=17


## 1. Recreate the fixed split

A fixed split makes a one-epoch baseline comparable across the class. Oxford-IIIT Pet trimaps become `1 = pet`, `0 = background`, and `255 = ignored boundary`.

In [ ]:
# Set OXFORD_PET_ROOT to the directory containing oxford-iiit-pet/ when data is pre-cached.
DATA_ROOT = Path(os.environ.get("OXFORD_PET_ROOT", "data"))
base = OxfordIIITPet(root=DATA_ROOT, split="trainval", target_types="segmentation", download=True)
permutation = torch.randperm(len(base), generator=torch.Generator().manual_seed(SEED)).tolist()
train_indices, val_indices = permutation[:64], permutation[64:88]
assert len(train_indices) == 64 and len(val_indices) == 24
assert not set(train_indices).intersection(val_indices)
print(f"fixed split: train={len(train_indices)}, validation={len(val_indices)}, overlap=0")
print("first five train indices:", train_indices[:5])

100%|██████████| 792M/792M [00:26<00:00, 29.9MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 14.8MB/s]


fixed split: train=64, validation=24, overlap=0
first five train indices: [2479, 1638, 733, 859, 670]


In [ ]:
def remap_trimap(raw_mask):
    raw = torch.as_tensor(np.asarray(raw_mask, dtype=np.uint8), dtype=torch.long)
    target = torch.full_like(raw, 255)
    target[raw == 1] = 1  # pet
    target[raw == 2] = 0  # background
    return target

def paired_transform(image, raw_mask):
    image = TF.resize(image, IMAGE_SIZE, interpolation=InterpolationMode.BILINEAR, antialias=True)
    raw_mask = TF.resize(raw_mask, IMAGE_SIZE, interpolation=InterpolationMode.NEAREST)
    image = TF.normalize(TF.to_tensor(image), mean=MEAN, std=STD)
    return image, remap_trimap(raw_mask)

class PetSubset(Dataset):
    def __init__(self, dataset, indices): self.dataset, self.indices = dataset, list(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        image, trimap = self.dataset[self.indices[i]]
        return paired_transform(image, trimap)

train_loader = DataLoader(PetSubset(base, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(PetSubset(base, val_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
images, masks = next(iter(train_loader))
assert images.shape == (BATCH_SIZE, 3, 128, 128) and masks.shape == (BATCH_SIZE, 128, 128)
assert set(torch.unique(masks).tolist()) <= {0, 1, 255}
print("batch contract:", tuple(images.shape), tuple(masks.shape), torch.unique(masks).tolist())

batch contract: (8, 3, 128, 128) (8, 128, 128) [0, 1, 255]


## 2. Replace the output head

The pretrained backbone already extracts useful visual features. Its original classifier was trained for VOC classes, so both LR-ASPP classifier branches must be replaced with two-output convolutions: background and pet.

In [ ]:
weights = LRASPP_MobileNet_V3_Large_Weights.DEFAULT
model = lraspp_mobilenet_v3_large(weights=weights)
model.classifier.low_classifier = nn.Conv2d(model.classifier.low_classifier.in_channels, 2, kernel_size=1)
model.classifier.high_classifier = nn.Conv2d(model.classifier.high_classifier.in_channels, 2, kernel_size=1)
model = model.to(device)
with torch.no_grad(): logits = model(images.to(device))["out"]
print("logits:", tuple(logits.shape))
assert logits.shape == (BATCH_SIZE, 2, 128, 128)


Downloading: "https://download.pytorch.org/models/lraspp_mobilenet_v3_large-d234d4ea.pth" to /root/.cache/torch/hub/checkpoints/lraspp_mobilenet_v3_large-d234d4ea.pth


100%|██████████| 12.5M/12.5M [00:00<00:00, 64.1MB/s]


logits: (8, 2, 128, 128)


## 3. Freeze intentionally

Freezing is a choice, not a default: the backbone keeps its learned image features while the small new head learns the two course classes. The optimizer should receive only parameters whose `requires_grad` is true.

In [ ]:
for parameter in model.backbone.parameters():
    parameter.requires_grad = False
trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
optimizer = torch.optim.Adam(trainable, lr=1e-3)
print("trainable names:", trainable_names)
assert trainable_names and all(name.startswith("classifier.") for name in trainable_names)
assert all(not parameter.requires_grad for parameter in model.backbone.parameters())


trainable names: ['classifier.cbr.0.weight', 'classifier.cbr.1.weight', 'classifier.cbr.1.bias', 'classifier.scale.1.weight', 'classifier.low_classifier.weight', 'classifier.low_classifier.bias', 'classifier.high_classifier.weight', 'classifier.high_classifier.bias']


## 4. Measure a pre-training baseline

Cross-entropy ignores `255` borders. The metric accumulator counts valid pixels over the whole validation split, instead of averaging per-image IoUs.

In [ ]:
def evaluate(model, loader, keep_predictions=False):
    model.eval(); total_loss = 0.0; valid_pixels = correct = intersection = union = 0
    saved_images, saved_targets, saved_predictions, saved_logits = [], [], [], []
    with torch.no_grad():
        for batch_images, targets in loader:
            batch_images, targets = batch_images.to(device), targets.to(device)
            logits = model(batch_images)["out"]
            total_loss += F.cross_entropy(logits, targets, ignore_index=255, reduction="sum").item()
            predictions = logits.argmax(dim=1)
            valid = targets != 255
            valid_pixels += valid.sum().item(); correct += ((predictions == targets) & valid).sum().item()
            intersection += ((predictions == 1) & (targets == 1) & valid).sum().item()
            union += (((predictions == 1) | (targets == 1)) & valid).sum().item()
            if keep_predictions:
                saved_images.append(batch_images.cpu()); saved_targets.append(targets.cpu())
                saved_predictions.append(predictions.cpu()); saved_logits.append(logits.cpu())
    result = {"loss": total_loss / max(valid_pixels, 1), "pixel_accuracy": correct / max(valid_pixels, 1), "pet_iou": intersection / max(union, 1)}
    if keep_predictions:
        result.update(images=torch.cat(saved_images), targets=torch.cat(saved_targets), predictions=torch.cat(saved_predictions), logits=torch.cat(saved_logits))
    return result

before = evaluate(model, val_loader)
print({key: round(value, 4) for key, value in before.items()})
assert all(np.isfinite(value) for value in before.values())

{'loss': 0.6477, 'pixel_accuracy': 0.6711, 'pet_iou': 0.0764}


## 5. Fine-tune one epoch and save the handoff artifact

The artifact contains model weights, seed, split indices, configuration, validation logits, predictions, and scores. Session 5 should load `artifacts/session4_baseline.pt`; if training is unavailable, it should instead use the instructor-provided `artifacts/instructor_baseline.pt`.

In [ ]:
model.train(); epoch_losses = []
for batch_images, targets in train_loader:
    batch_images, targets = batch_images.to(device), targets.to(device)
    optimizer.zero_grad()
    logits = model(batch_images)["out"]
    loss = F.cross_entropy(logits, targets, ignore_index=255)
    loss.backward(); optimizer.step()
    epoch_losses.append(loss.item())
mean_train_loss = float(np.mean(epoch_losses))
after = evaluate(model, val_loader, keep_predictions=True)
artifact = {
    "artifact_version": 1, "seed": SEED, "image_size": IMAGE_SIZE, "epochs": EPOCHS,
    "train_indices": train_indices, "val_indices": val_indices, "mean_train_loss": mean_train_loss,
    "metrics": {key: value for key, value in after.items() if isinstance(value, float)},
    "model_state_dict": {key: value.cpu() for key, value in model.state_dict().items()},
    "validation_images": after["images"], "validation_targets": after["targets"],
    "validation_predictions": after["predictions"], "validation_logits": after["logits"],
}
torch.save(artifact, BASELINE_PATH)
print(f"one epoch mean loss={mean_train_loss:.4f}")
print("saved Session 5 handoff:", BASELINE_PATH.resolve())
print({key: round(value, 4) for key, value in artifact["metrics"].items()})
assert BASELINE_PATH.exists() and np.isfinite(mean_train_loss)

one epoch mean loss=0.2769
saved Session 5 handoff: /content/artifacts/session4_baseline.pt
{'loss': 0.3403, 'pixel_accuracy': 0.8403, 'pet_iou': 0.6291}


In [ ]:
handoff_path = BASELINE_PATH if BASELINE_PATH.exists() else PREPARED_BASELINE
if handoff_path.exists():
    saved = torch.load(handoff_path, map_location="cpu", weights_only=False)
    print(f"Session 5 should load: {handoff_path.resolve()}")
    print("saved seed/epochs:", saved["seed"], saved["epochs"])
else:
    print("No prepared fallback yet. Ask the instructor for artifacts/instructor_baseline.pt.")

Session 5 should load: /content/artifacts/session4_baseline.pt
saved seed/epochs: 17 1
